3.3.1 Attention mechanisms - its kind of RNN for generating one token/text

Simple self attention mechanism without trainable weights

In [ ]:
import torch
inputs = torch.tensor(
    [[0.43,0.15,0.89], # your
    [0.55, 0.87, 0.66], # journey
    [0.57,0.85,0.64], # starts
    [0.22,0.58,0.33], # with
    [0.77, 0.25, 0.10], # one
    [0.05, 0.8,0.55]] # step
)

c:\Srujith\Fun\LLM\.venv\Lib\site-packages\torch\_subclasses\functional_tensor.py:362: UserWarning: Failed to initialize NumPy: No module named 'numpy' (Triggered internally at C:\actions-runner\_work\pytorch\pytorch\torch\csrc\utils\tensor_numpy.cpp:84.)
  cpu = _conversion_method_template(device=torch.device("cpu"))


In [ ]:
input_query = inputs[1] #consider one tensor at a time(python indexes at 0)
input_query

tensor([0.5500, 0.8700, 0.6600])

In [5]:
input_1 = inputs[0]
input_1

tensor([0.4300, 0.1500, 0.8900])

In [ ]:
torch.dot(input_query,input_1) # element wise dot product of these 2 input vectors

tensor(0.9544)

Its still not the attention weights(the ones used for getting context vector) we get the intermediate weights called - attentionscores(these are not yet normalized they can be greatet than 1)

In [6]:
res = 0.
i = 3
for idx, ele in enumerate(inputs[i]):
    res += inputs[i][idx]*input_query[idx]

res

tensor(0.8434)

In [8]:
i = 2
res = torch.dot(inputs[i], input_query)
res

tensor(1.4754)

In [21]:
inputs.shape[0]

6

In [23]:
query = inputs[0]
attn_scores_2 = torch.empty(inputs.shape[0])
for i, x_i in enumerate(inputs):
    attn_scores_2[i] = torch.dot(x_i,input_query) # dot product using pytorch just efficient than other

print(attn_scores_2)

tensor([0.9544, 1.4950, 1.4754, 0.8434, 0.7070, 1.0865])


In [ ]:
attn_weights_2_temp = attn_scores_2/attn_scores_2.sum()
attn_weights_2_temp  # kind of normalized here the attn weights, u can use this or u can use softmax

tensor([0.1455, 0.2278, 0.2249, 0.1285, 0.1077, 0.1656])

In [26]:
attn_weights_2_temp.sum()

tensor(1.0000)

In [ ]:
def softmax_naive(x):
    return torch.exp(x)/torch.exp(x).sum(dim=0)
softmax_naive(attn_scores_2) #softmax is numerically unstable at some cases, see these 2 normalized values are not the same. but roughly the same

tensor([0.1385, 0.2379, 0.2333, 0.1240, 0.1082, 0.1581])

In [ ]:
attn_weights_2 = torch.softmax(attn_scores_2,dim=0) #can use the pytorch version of this, this is more stable that

tensor([0.1385, 0.2379, 0.2333, 0.1240, 0.1082, 0.1581])

In [29]:
query = inputs[2] # 2nd input token is the query
context_vec_2 = torch.zeros(query.shape)
for i,x_i in enumerate(inputs):
    print(f"{attn_weights_2[i]}---->{inputs[i]}")
    context_vec_2 += attn_weights_2_temp[i]*x_i
    

print(context_vec_2)

0.13854756951332092---->tensor([0.4300, 0.1500, 0.8900])
0.2378913015127182---->tensor([0.5500, 0.8700, 0.6600])
0.23327402770519257---->tensor([0.5700, 0.8500, 0.6400])
0.12399158626794815---->tensor([0.2200, 0.5800, 0.3300])
0.10818186402320862---->tensor([0.7700, 0.2500, 0.1000])
0.15811361372470856---->tensor([0.0500, 0.8000, 0.5500])
tensor([0.4355, 0.6451, 0.5680])


This is the context vector, use the weights -> compute weighted sum. its like one attention weight multiply with the word vector -> do for all the words(input vectors) -> compute a tensor(vector) using all those attnetion weighted vectors. thats why it is called self- attnetion
coz - we use the index positon(word at hand) to compute all the weghts around.

now we did for one word position, we need do it for all the words at once

3.3.2 a simple self atttenton mechnasinm wihtout trainable weights- 
we stil dont use the wegihts to train the model, they are just computed not for training